# Notebook 67 — Policy Geometry

**Confidence-Scheduled Decoding**  
**Position:** Notebook 61 Telemetry Dataset → **Notebook 67 Policy Geometry** → Notebook 71 Offline Evaluation

## Engineering statement

**Telemetry specifies admissible policy geometry.**

Notebook 61 produces telemetry rows. Notebook 67 infers the geometry of admissible actions in decoding state space. Notebook 71 tests whether the inferred policy is admissible under offline replay.

The important object is not the classifier. The important object is the state partition:

\[
s_t = (c_t, e_t, m_t, r_t, b_t, v_t)
\]

where:

- \(c_t\): confidence
- \(e_t\): entropy
- \(m_t\): margin
- \(r_t\): risk score
- \(b_t\): latency budget
- \(v_t\): verifier disagreement

The policy maps state to action:

\[
\pi(s_t) \rightarrow a_t
\]

with:

\[
a_t \in \{\text{continue}, \text{deepen}, \text{verify}, \text{stop}, \text{fallback}\}.
\]

Notebook 67 estimates the policy geometry. Notebook 71 evaluates it.

## 1. Setup

All figures are written to `figures/`.

This notebook is self-contained. In the repo workflow, replace the synthetic telemetry cell with the Notebook 61 telemetry export.

In [ ]:
from pathlib import Path
import json
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

ROOT = Path.cwd()
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"

for d in [FIG_DIR, DATA_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(67)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.22

print("Notebook 67 refined ready")
print("Figures:", FIG_DIR.resolve())

## 2. Telemetry schema

Notebook 61 emits decision-point rows. Notebook 67 consumes those rows as state samples.

The policy observes only fields available at the current decoding step. Future reward is exported for Notebook 71 replay, not used as a policy feature.

In [ ]:
telemetry_schema = pd.DataFrame([
    ("request_id", "logical request identifier"),
    ("step", "decode step / policy decision index"),
    ("domain", "task family"),
    ("confidence", "model confidence proxy; larger is safer"),
    ("entropy", "next-token uncertainty proxy; larger is less certain"),
    ("margin", "top-1 minus top-2 probability margin"),
    ("risk_score", "external/user/system risk score"),
    ("latency_budget_ms", "remaining request latency budget"),
    ("budget_pressure", "normalized pressure induced by remaining budget"),
    ("tokens_so_far", "generated tokens before this decision"),
    ("cost_so_far", "normalized accumulated compute cost"),
    ("verifier_disagreement", "checker/model disagreement signal"),
    ("chosen_action", "observed or hindsight policy label"),
    ("reward", "offline scalar utility used later by Notebook 71"),
], columns=["field", "meaning"])

telemetry_schema

## 3. Synthetic Notebook-61-like telemetry

The synthetic generator is designed to produce visible policy geometry.

Qualitative rules:

- high confidence and low risk → **continue**
- low confidence or high entropy with enough budget → **deepen**
- high risk or high disagreement → **verify**
- high confidence with budget pressure → **stop**
- low confidence and exhausted budget → **fallback**

This refined version strengthens budget coupling and preserves all five action regions.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

domains = np.array(["math", "code", "qa", "safety", "long_context"])
n = 4600

domain = RNG.choice(domains, size=n, p=[0.24, 0.20, 0.26, 0.14, 0.16])

domain_risk = {
    "math": 0.22,
    "code": 0.40,
    "qa": 0.31,
    "safety": 0.78,
    "long_context": 0.52,
}

domain_entropy_shift = {
    "math": -0.07,
    "code": 0.10,
    "qa": 0.00,
    "safety": 0.18,
    "long_context": 0.24,
}

step = RNG.integers(1, 96, size=n)
tokens_so_far = step + RNG.poisson(18, size=n)
latency_budget_ms = np.clip(RNG.normal(900 - 5.8 * step, 220, size=n), 60, 1450)

base_confidence = sigmoid(RNG.normal(0.75, 1.0, size=n) - 0.012 * step)
confidence = np.clip(
    base_confidence - np.vectorize(domain_entropy_shift.get)(domain) * 0.20,
    0.02,
    0.995,
)

entropy = np.clip(
    1.55
    - 1.25 * confidence
    + np.vectorize(domain_entropy_shift.get)(domain)
    + RNG.normal(0, 0.16, size=n),
    0.05,
    2.2,
)

margin = np.clip(
    0.70 * confidence - 0.22 * entropy + RNG.normal(0.06, 0.12, size=n),
    0.00,
    0.95,
)

risk_score = np.clip(
    np.vectorize(domain_risk.get)(domain)
    + RNG.normal(0, 0.10, size=n)
    + 0.18 * (entropy > 1.10),
    0,
    1,
)

cost_so_far = np.clip(
    0.0017 * tokens_so_far + 0.0009 * step + RNG.normal(0.02, 0.015, size=n),
    0,
    1,
)

verifier_disagreement = np.clip(
    0.16
    + 0.42 * (entropy > 1.00)
    + 0.24 * (risk_score > 0.65)
    + RNG.normal(0, 0.13, size=n),
    0,
    1,
)

budget_pressure = 1 - (
    (latency_budget_ms - latency_budget_ms.min())
    / (latency_budget_ms.max() - latency_budget_ms.min())
)

actions = []

for c, e, m, r, b, v in zip(
    confidence,
    entropy,
    margin,
    risk_score,
    budget_pressure,
    verifier_disagreement,
):
    # Verifier gate: risk and disagreement can override ordinary confidence logic.
    if r > 0.74 or v > 0.74 or (r + 0.8 * v > 1.15):
        action = "verify"

    # Budget deformation: under low budget, stop/fallback expand.
    elif b > 0.74 and c > 0.55 and m > 0.18:
        action = "stop"
    elif b > 0.78 and c < 0.46:
        action = "fallback"

    # High budget admits deeper reasoning.
    elif b < 0.50 and (c < 0.54 or e > 1.02 or m < 0.20):
        action = "deepen"

    # Moderate budget still deepens difficult states.
    elif c < 0.38 or (e > 1.18 and m < 0.26):
        action = "deepen"

    else:
        action = "continue"

    # Noisy telemetry / imperfect labels.
    if RNG.random() < 0.075:
        action = RNG.choice(
            ["continue", "deepen", "verify", "stop", "fallback"],
            p=[0.36, 0.26, 0.18, 0.14, 0.06],
        )

    actions.append(action)

utility_by_action = {
    "continue": 0.74,
    "deepen": 0.80,
    "verify": 0.77,
    "stop": 0.70,
    "fallback": 0.50,
}

reward = np.array([utility_by_action[a] for a in actions])
reward += 0.18 * confidence + 0.10 * margin
reward -= 0.16 * entropy + 0.11 * cost_so_far + 0.08 * budget_pressure
reward += RNG.normal(0, 0.045, size=n)
reward = np.clip(reward, 0, 1)

telemetry = pd.DataFrame({
    "request_id": [f"req_{i//8:05d}" for i in range(n)],
    "step": step,
    "domain": domain,
    "confidence": confidence,
    "entropy": entropy,
    "margin": margin,
    "risk_score": risk_score,
    "latency_budget_ms": latency_budget_ms,
    "budget_pressure": budget_pressure,
    "tokens_so_far": tokens_so_far,
    "cost_so_far": cost_so_far,
    "verifier_disagreement": verifier_disagreement,
    "chosen_action": actions,
    "reward": reward,
})

telemetry.to_csv(DATA_DIR / "notebook67_policy_geometry_telemetry.csv", index=False)
telemetry.head()

## 4. Dataset checks

The first check is not model accuracy. The first check is whether the telemetry covers enough policy states to infer geometry.

In [ ]:
summary = telemetry.describe(include="all").transpose()
summary.to_csv(DATA_DIR / "notebook67_telemetry_summary.csv")
summary

In [ ]:
action_order = ["continue", "deepen", "verify", "stop", "fallback"]

fig, ax = plt.subplots(figsize=(8, 4.8))
telemetry["chosen_action"].value_counts().reindex(action_order).plot(kind="bar", ax=ax)
ax.set_title("Notebook 67 — policy action distribution", fontsize=16)
ax.set_xlabel("Action")
ax.set_ylabel("Telemetry rows")
fig.tight_layout()
fig.savefig(FIG_DIR / "67_00_action_distribution.png", dpi=180)
plt.show()

## 5. Figure 1 — State manifold

Confidence alone does not specify the policy. The useful object is the geometry of confidence, entropy, margin, risk, budget, and verifier disagreement.

This plot is the visible state manifold.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Draw the dense continue class first and more faintly.
plot_order = ["continue", "deepen", "fallback", "stop", "verify"]
alpha_by_action = {
    "continue": 0.10,
    "deepen": 0.18,
    "fallback": 0.22,
    "stop": 0.22,
    "verify": 0.26,
}
size_by_action = {
    "continue": 9,
    "deepen": 12,
    "fallback": 16,
    "stop": 16,
    "verify": 18,
}

for action in plot_order:
    group = telemetry[telemetry["chosen_action"] == action]
    ax.scatter(
        group["entropy"],
        group["confidence"],
        s=size_by_action[action],
        alpha=alpha_by_action[action],
        label=action,
    )

ax.set_title("67.01 — State manifold", fontsize=18)
ax.set_xlabel("Entropy", fontsize=14)
ax.set_ylabel("Confidence", fontsize=14)
ax.legend(markerscale=2, frameon=True, title="Observed action")
fig.tight_layout()
fig.savefig(FIG_DIR / "67_01_state_manifold.png", dpi=180)
plt.show()

## 6. Figure 2 — Latent policy manifold

The telemetry state has more than two dimensions.

A PCA projection of confidence, entropy, margin, risk, budget, cost, and disagreement shows how the observed actions occupy a lower-dimensional policy manifold.

This is not used as the policy. It is a visualization of the geometry that Notebook 67 estimates.

In [ ]:
pca_features = [
    "confidence",
    "entropy",
    "margin",
    "risk_score",
    "latency_budget_ms",
    "budget_pressure",
    "cost_so_far",
    "verifier_disagreement",
]

scaler = StandardScaler()
Z_scaled = scaler.fit_transform(telemetry[pca_features])

pca = PCA(n_components=2, random_state=67)
latent = pca.fit_transform(Z_scaled)

telemetry["latent_1"] = latent[:, 0]
telemetry["latent_2"] = latent[:, 1]

fig, ax = plt.subplots(figsize=(10, 6))

for action in plot_order:
    group = telemetry[telemetry["chosen_action"] == action]
    ax.scatter(
        group["latent_1"],
        group["latent_2"],
        s=size_by_action[action],
        alpha=alpha_by_action[action],
        label=action,
    )

ax.set_title("67.02 — Latent policy manifold", fontsize=18)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)", fontsize=13)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)", fontsize=13)
ax.legend(markerscale=2, frameon=True, title="Observed action")
fig.tight_layout()
fig.savefig(FIG_DIR / "67_02_latent_policy_manifold.png", dpi=180)
plt.show()

## 7. Figure 3 — Smooth conceptual policy geometry

Before fitting any estimator, we can state the intended engineering geometry.

The admissibility score is:

\[
A(a \mid s) =
U(a, s)
- \lambda_{cost} C(a, s)
- \lambda_{risk} R(a, s)
- \lambda_{latency} L(a, s)
\]

and the policy approximates:

\[
\hat{\pi}(s) = \arg\max_a A(a \mid s)
\]

The conceptual geometry below is not learned. It is a smooth score-based design target.

In [ ]:
# Smooth conceptual policy geometry.
conf = np.linspace(0.02, 0.99, 320)
ent = np.linspace(0.02, 2.10, 320)
C, E = np.meshgrid(conf, ent)

def gaussian2(x, y, mx, my, sx, sy):
    return np.exp(-(((x - mx) / sx) ** 2 + ((y - my) / sy) ** 2) / 2)

continue_score = (
    1.35 * gaussian2(C, E, 0.76, 0.72, 0.30, 0.42)
    + 0.35 * C
    - 0.18 * E
)

deepen_score = (
    1.25 * gaussian2(C, E, 0.36, 1.55, 0.34, 0.55)
    + 0.35 * E
    - 0.18 * C
)

verify_score = (
    1.35 * gaussian2(C, E, 0.58, 1.15, 0.18, 0.28)
    + 0.15 * E
)

stop_score = (
    1.35 * gaussian2(C, E, 0.88, 0.25, 0.16, 0.22)
    + 0.32 * C
    - 0.45 * E
)

fallback_score = (
    1.25 * gaussian2(C, E, 0.14, 1.75, 0.20, 0.35)
    + 0.25 * E
    - 0.35 * C
)

scores = np.stack([continue_score, deepen_score, verify_score, stop_score, fallback_score], axis=0)
Z_smooth = np.argmax(scores, axis=0)

labels = ["continue", "deepen", "verify", "stop", "fallback"]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    Z_smooth,
    origin="lower",
    aspect="auto",
    extent=[conf.min(), conf.max(), ent.min(), ent.max()],
)

ax.contour(C, E, Z_smooth, levels=np.arange(-0.5, 5, 1), linewidths=0.8, alpha=0.35)

ax.set_title("67.03 — Smooth conceptual policy geometry", fontsize=18)
ax.set_xlabel("Confidence", fontsize=14)
ax.set_ylabel("Entropy", fontsize=14)

cbar = fig.colorbar(im, ax=ax, ticks=list(range(len(labels))))
cbar.ax.set_yticklabels(labels)

ax.text(0.82, 0.30, "stop", ha="center", va="center", fontsize=12)
ax.text(0.72, 0.82, "continue", ha="center", va="center", fontsize=12)
ax.text(0.18, 1.72, "fallback", ha="center", va="center", fontsize=12)
ax.text(0.36, 1.42, "deepen", ha="center", va="center", fontsize=12)
ax.text(0.58, 1.16, "verify", ha="center", va="center", fontsize=12)

fig.tight_layout()
fig.savefig(FIG_DIR / "67_03_smooth_policy_geometry.png", dpi=180)
plt.show()

## 8. Learn the policy geometry

The estimator is an implementation detail.

Here we use a random forest because it gives nonlinear boundaries while remaining easy to inspect. The output is an estimated action surface and action probabilities for Notebook 71.

In [ ]:
features = [
    "step",
    "confidence",
    "entropy",
    "margin",
    "risk_score",
    "latency_budget_ms",
    "budget_pressure",
    "tokens_so_far",
    "cost_so_far",
    "verifier_disagreement",
]

X_numeric = telemetry[features].copy()
X_domain = pd.get_dummies(telemetry["domain"], prefix="domain")
X = pd.concat([X_numeric, X_domain], axis=1)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(telemetry["chosen_action"])

X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X,
    y,
    telemetry.index,
    test_size=0.25,
    random_state=67,
    stratify=y,
)

policy = RandomForestClassifier(
    n_estimators=240,
    max_depth=10,
    min_samples_leaf=8,
    class_weight="balanced_subsample",
    random_state=67,
    n_jobs=-1,
)

policy.fit(X_train, y_train)

pred = policy.predict(X_test)
proba = policy.predict_proba(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "macro_f1": float(f1_score(y_test, pred, average="macro")),
    "log_loss": float(log_loss(y_test, proba)),
    "classes": label_encoder.classes_.tolist(),
}

metrics

## 9. Figure 4 — Learned admissible regions with telemetry overlay

This surface is the learned estimate of the policy geometry.

The overlay matters: it distinguishes regions supported by telemetry from regions that are mostly extrapolation.

In [ ]:
grid_n = 150
conf_grid = np.linspace(0.04, 0.98, grid_n)
ent_grid = np.linspace(0.05, 2.05, grid_n)

template = X.median(numeric_only=True).to_dict()
for col in X.columns:
    if col.startswith("domain_"):
        template[col] = 0
template["domain_qa"] = 1

rows = []
for e in ent_grid:
    for c in conf_grid:
        row = template.copy()
        row["confidence"] = c
        row["entropy"] = e
        row["margin"] = np.clip(0.70 * c - 0.22 * e + 0.06, 0, 0.95)
        rows.append(row)

grid = pd.DataFrame(rows)[X.columns]
grid_pred = policy.predict(grid)
grid_actions = label_encoder.inverse_transform(grid_pred).reshape(grid_n, grid_n)

action_to_int = {a: i for i, a in enumerate(label_encoder.classes_)}
Z_learned = np.vectorize(action_to_int.get)(grid_actions)

fig, ax = plt.subplots(figsize=(10.5, 6.2))
im = ax.imshow(
    Z_learned,
    origin="lower",
    aspect="auto",
    extent=[conf_grid.min(), conf_grid.max(), ent_grid.min(), ent_grid.max()],
    alpha=0.88,
)

# Telemetry overlay: use a sampled subset to keep the plot readable.
sample = telemetry.sample(n=min(1200, len(telemetry)), random_state=67)
ax.scatter(
    sample["confidence"],
    sample["entropy"],
    s=7,
    alpha=0.20,
    c="white",
    edgecolors="black",
    linewidths=0.15,
)

ax.set_title("67.04 — Learned regions with telemetry overlay", fontsize=18)
ax.set_xlabel("Confidence", fontsize=14)
ax.set_ylabel("Entropy", fontsize=14)

cbar = fig.colorbar(im, ax=ax, ticks=list(action_to_int.values()))
cbar.ax.set_yticklabels(list(action_to_int.keys()))

fig.tight_layout()
fig.savefig(FIG_DIR / "67_04_learned_regions_overlay.png", dpi=180)
plt.show()

## 10. Figure 5 — Budget deformation

Budget changes the geometry.

High budget admits deeper reasoning. Low budget expands stop/fallback behavior. The policy boundary should visibly deform.

In [ ]:
low_budget = float(np.percentile(telemetry["latency_budget_ms"], 8))
high_budget = float(np.percentile(telemetry["latency_budget_ms"], 92))

def surface_for_budget(latency_budget_value):
    rows = []
    budget_pressure_value = 1 - (
        (latency_budget_value - telemetry["latency_budget_ms"].min())
        / (telemetry["latency_budget_ms"].max() - telemetry["latency_budget_ms"].min())
    )

    for e in ent_grid:
        for c in conf_grid:
            row = template.copy()
            row["confidence"] = c
            row["entropy"] = e
            row["margin"] = np.clip(0.70 * c - 0.22 * e + 0.06, 0, 0.95)
            row["latency_budget_ms"] = latency_budget_value
            row["budget_pressure"] = budget_pressure_value
            rows.append(row)

    surface_df = pd.DataFrame(rows)[X.columns]
    surface_actions = label_encoder.inverse_transform(policy.predict(surface_df)).reshape(grid_n, grid_n)
    return np.vectorize(action_to_int.get)(surface_actions)

Z_high = surface_for_budget(high_budget)
Z_low = surface_for_budget(low_budget)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharex=True, sharey=True)

for ax, Z_budget, subtitle in [
    (axes[0], Z_high, f"high budget: {high_budget:.0f} ms"),
    (axes[1], Z_low, f"low budget: {low_budget:.0f} ms"),
]:
    im = ax.imshow(
        Z_budget,
        origin="lower",
        aspect="auto",
        extent=[conf_grid.min(), conf_grid.max(), ent_grid.min(), ent_grid.max()],
    )
    ax.set_title(subtitle, fontsize=15)
    ax.set_xlabel("Confidence", fontsize=13)

axes[0].set_ylabel("Entropy", fontsize=13)
fig.suptitle("67.05 — Budget deformation", fontsize=18, y=0.98)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), ticks=list(action_to_int.values()), fraction=0.046, pad=0.04)
cbar.ax.set_yticklabels(list(action_to_int.keys()))

fig.savefig(FIG_DIR / "67_05_budget_deformation.png", dpi=180, bbox_inches="tight")
plt.show()

## 11. Figure 6 — Verifier boundary

Risk and verifier disagreement are gates.

This figure is intentionally an engineering diagram rather than a learned classifier surface. It states the safety boundary that Notebook 71 should stress-test.

In [ ]:
x = np.linspace(0, 1, 300)
boundary = np.clip(0.74 - 0.52 * x + 0.10 * np.sin(2 * np.pi * x), 0.18, 0.82)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(x, boundary, linewidth=3)
ax.fill_between(x, boundary, 1, alpha=0.25)
ax.fill_between(x, 0, boundary, alpha=0.12)

ax.text(0.72, 0.82, "VERIFY", ha="center", va="center", fontsize=20)
ax.text(0.32, 0.24, "continue / deepen / stop", ha="center", va="center", fontsize=14)

ax.annotate(
    "disagreement lowers the risk threshold",
    xy=(0.70, boundary[np.searchsorted(x, 0.70)]),
    xytext=(0.36, 0.92),
    arrowprops={"arrowstyle": "->", "lw": 1.8},
    fontsize=12,
)

ax.set_title("67.06 — Verifier boundary", fontsize=18)
ax.set_xlabel("Verifier disagreement", fontsize=14)
ax.set_ylabel("Risk score", fontsize=14)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIG_DIR / "67_06_verifier_boundary.png", dpi=180)
plt.show()

## 12. Figure 7 — Policy emergence

This is the notebook in one diagram.

Telemetry rows define state samples. State samples form a manifold. The manifold is partitioned into admissible regions. The resulting policy candidate is tested by Notebook 71.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 10))
ax.axis("off")

box_style = dict(boxstyle="round,pad=0.55", facecolor="white", edgecolor="black", linewidth=1.5)

stages = [
    ("Notebook 61\\nTelemetry rows", 0.50, 0.88),
    ("State manifold\\nconfidence, entropy, risk, budget", 0.50, 0.70),
    ("Admissible geometry\\ncontinue / deepen / verify / stop / fallback", 0.50, 0.52),
    ("Estimated policy\\npolicy candidate", 0.50, 0.34),
    ("Notebook 71\\nOffline replay", 0.50, 0.16),
]

for text, x0, y0 in stages:
    ax.text(x0, y0, text, ha="center", va="center", fontsize=13, bbox=box_style)

for i in range(len(stages) - 1):
    x1, y1 = stages[i][1], stages[i][2]
    x2, y2 = stages[i + 1][1], stages[i + 1][2]
    ax.annotate("", xy=(x2, y2 + 0.07), xytext=(x1, y1 - 0.07), arrowprops=dict(arrowstyle="->", lw=2))

# small decorative telemetry cloud
cloud_x = 0.23 + 0.08 * RNG.normal(size=70)
cloud_y = 0.70 + 0.035 * RNG.normal(size=70)
ax.scatter(cloud_x, cloud_y, s=10, alpha=0.25)

ax.set_title("67.07 — Policy emergence", fontsize=20)
fig.savefig(FIG_DIR / "67_07_policy_emergence.png", dpi=180, bbox_inches="tight")
plt.show()

## 13. Figure 8 — Feature importance

The learned policy should be inspectable.

Permutation importance asks how much held-out performance falls when a feature is shuffled. The annotations group the top drivers by role.

In [ ]:
importance_result = permutation_importance(
    policy,
    X_test,
    y_test,
    n_repeats=8,
    random_state=67,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": importance_result.importances_mean,
    "importance_std": importance_result.importances_std,
}).sort_values("importance_mean", ascending=False)

importance.to_csv(DATA_DIR / "notebook67_policy_feature_importance.csv", index=False)

feature_roles = {
    "confidence": "state",
    "entropy": "uncertainty",
    "margin": "uncertainty",
    "risk_score": "safety",
    "latency_budget_ms": "resource",
    "budget_pressure": "resource",
    "verifier_disagreement": "consensus",
    "tokens_so_far": "progress",
    "cost_so_far": "resource",
    "step": "progress",
}

importance["role"] = importance["feature"].map(feature_roles).fillna("domain")
importance.to_csv(DATA_DIR / "notebook67_policy_feature_importance_with_roles.csv", index=False)

top = importance.head(12).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
ax.set_title("67.08 — Policy feature importance", fontsize=18)
ax.set_xlabel("Mean held-out accuracy decrease", fontsize=13)
ax.set_ylabel("Feature", fontsize=13)

role_lines = []
for _, row in importance.head(6).iterrows():
    role_lines.append(f"{row['feature']}: {row['role']}")
note = "Top roles:\\n" + "\\n".join(role_lines)

ax.text(
    0.98,
    0.08,
    note,
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black", alpha=0.85),
)

fig.tight_layout()
fig.savefig(FIG_DIR / "67_08_feature_importance.png", dpi=180)
plt.show()

importance.head(12)

## 14. Figure 9 — Held-out diagnostics

Notebook 67 uses held-out diagnostics only as a sanity check. Notebook 71 will perform replay, baselines, counterfactual policy scoring, cost-quality curves, and safety/regret analysis.

In [ ]:
report = classification_report(
    y_test,
    pred,
    target_names=label_encoder.classes_,
    output_dict=True,
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv(DATA_DIR / "notebook67_policy_classification_report.csv")
report_df

In [ ]:
cm = confusion_matrix(y_test, pred)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm)

ax.set_title("67.09 — Learned policy confusion matrix", fontsize=18)
ax.set_xticks(range(len(label_encoder.classes_)))
ax.set_yticks(range(len(label_encoder.classes_)))
ax.set_xticklabels(label_encoder.classes_, rotation=35, ha="right")
ax.set_yticklabels(label_encoder.classes_)
ax.set_xlabel("Predicted action", fontsize=13)
ax.set_ylabel("Observed / target action", fontsize=13)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "67_09_confusion_matrix.png", dpi=180)
plt.show()

metrics

## 15. Notebook 71 handoff

Notebook 67 exports the policy replay table.

Notebook 71 should evaluate the learned action against baselines:

- static confidence threshold
- entropy threshold
- always continue
- always verify
- expert/hindsight policy
- learned policy

The handoff table includes action probabilities so Notebook 71 can compute propensity-aware diagnostics if needed.

In [ ]:
test_rows = telemetry.loc[test_idx].copy()
test_rows["learned_action"] = label_encoder.inverse_transform(pred)
test_rows["learned_action_probability"] = proba.max(axis=1)

handoff = test_rows[
    [
        "request_id",
        "step",
        "domain",
        "confidence",
        "entropy",
        "margin",
        "risk_score",
        "latency_budget_ms",
        "budget_pressure",
        "tokens_so_far",
        "cost_so_far",
        "verifier_disagreement",
        "chosen_action",
        "reward",
        "learned_action",
        "learned_action_probability",
    ]
].copy()

for i, cls in enumerate(label_encoder.classes_):
    handoff[f"p_{cls}"] = proba[:, i]

handoff.to_csv(DATA_DIR / "notebook67_to_71_offline_eval_handoff.csv", index=False)
handoff.head()

In [ ]:
policy_card = {
    "notebook": "67_policy_geometry",
    "title": "Policy Geometry",
    "repo": "github.com/thinkthoughts/confidence-scheduled-decoding",
    "position": {
        "previous": "61_telemetry_dataset",
        "current": "67_policy_geometry",
        "next": "71_offline_evaluation",
    },
    "statement": "Telemetry specifies admissible policy geometry.",
    "features": X.columns.tolist(),
    "feature_roles": feature_roles,
    "actions": label_encoder.classes_.tolist(),
    "metrics": metrics,
    "figures": sorted(p.name for p in FIG_DIR.glob("67_*.png")),
    "handoff_file": "data/notebook67_to_71_offline_eval_handoff.csv",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "notebook_71_questions": [
        "Does the learned policy improve reward at fixed cost?",
        "Does it reduce unnecessary verifier calls?",
        "Does it preserve safety behavior on high-risk rows?",
        "Where does it fail under domain shift?",
        "Which telemetry fields would repair the failures?",
    ],
}

with open(ARTIFACT_DIR / "notebook67_policy_card.json", "w") as f:
    json.dump(policy_card, f, indent=2)

policy_card

## 16. Artifact summary

Notebook 67 is an engineering component. Its outputs are intended for the next notebook and for repo-level documentation.

In [ ]:
artifact_summary = pd.DataFrame([
    ("figures/*.png", "report / README / visual review"),
    ("data/notebook67_policy_geometry_telemetry.csv", "local reproducibility"),
    ("data/notebook67_to_71_offline_eval_handoff.csv", "Notebook 71"),
    ("data/notebook67_policy_feature_importance.csv", "diagnostics"),
    ("data/notebook67_policy_classification_report.csv", "diagnostics"),
    ("artifacts/notebook67_policy_card.json", "README / repo metadata"),
], columns=["artifact", "consumer"])

artifact_summary.to_csv(DATA_DIR / "notebook67_artifact_summary.csv", index=False)
artifact_summary

## 17. Bridge table

This table keeps the series aligned.

In [ ]:
bridge = pd.DataFrame([
    {
        "from_notebook": 61,
        "artifact": "telemetry rows",
        "to_notebook": 67,
        "role": "infer admissible policy geometry over decoding states",
    },
    {
        "from_notebook": 67,
        "artifact": "policy card + action probabilities + handoff CSV",
        "to_notebook": 71,
        "role": "evaluate learned policy under offline replay",
    },
])

bridge.to_csv(DATA_DIR / "notebook67_bridge_table.csv", index=False)
bridge

## 18. Summary

Notebook 67 defines the policy-geometry bridge.

**Result:** telemetry rows become an estimated admissible-action manifold.

**Constraint:** this notebook does not claim deployment readiness.

**Next:** Notebook 71 performs offline evaluation and asks whether the learned policy is admissible under replay.

\[
\mathcal{T}
\rightarrow
\mathcal{S}
\rightarrow
\mathcal{G}
\rightarrow
\hat{\pi}
\rightarrow
\mathcal{E}
\]

where:

- \(\mathcal{T}\): telemetry
- \(\mathcal{S}\): state manifold
- \(\mathcal{G}\): admissible geometry
- \(\hat{\pi}\): learned policy
- \(\mathcal{E}\): offline evaluation

## 19. Final Colab download cell

Run this cell at the end of the notebook to package the generated figures, data, artifacts, and notebook.

In [ ]:
zip_name = "notebook67_policy_geometry_outputs.zip"
zip_path = ROOT / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, DATA_DIR, ARTIFACT_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.relative_to(ROOT))
    notebook_file = ROOT / "67_policy_geometry.ipynb"
    if notebook_file.exists():
        z.write(notebook_file, "67_policy_geometry.ipynb")

print(f"Wrote {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))